# Lektion 20: Testing, Evaluierung & dein Abschluss



| | |
|---|---|
| **Zeit** | ~60 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 01–19 |
| **Du baust** | Eigene Tests für dein System, eine kriterienbasierte Evaluierung – und deine Abschluss-Reflexion |

## 🎯 Nach dieser Lektion kannst du:

- Unit- und Integrationstests für deine Bausteine schreiben

- erklären, warum Mocks die *Mechanik*, aber nicht die *Qualität* echter LLM-Antworten prüfen

- eine kriterienbasierte **Evaluierung** (Erfolgsquote) durchführen

- Kosten-Regeln benennen (Loop-Limits, kleine Modelle, Caching)

- die drei Architekturen mit eigenen Worten vergleichen (Abschluss!)

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [ ]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


## 🧠 Konzept: „Ich habe es einmal probiert" ist kein Test

**WARUM:** KI-Verhalten ist nicht deterministisch – einmal laufen lassen beweist gar nichts. Und wenn du später einen Prompt änderst: Wer merkt, dass die Antwortqualität eingebrochen ist?

**WAS – die zwei Prüfarten:**
- **Unittest** = Rechenaufgabe mit eindeutiger Lösung (`30 Urlaubstage == 30`). Prüft **Logik**.
- **Eval** = Bewertungsraster („Enthält die Antwort die richtige Quelle?"). Prüft **Eigenschaften** von Antworten.

**Metapher:** Mocks prüfen, ob der Motor sich dreht (Mechanik). Evals prüfen, ob das Auto gut fährt (Qualität) – letzteres braucht mit echtem Modell mehr als eine Probefahrt.

In [ ]:
# Unit-Test-Stil: wir pruefen Logik, die wir selbst gebaut haben.
import sys
from pathlib import Path
sys.path.insert(0, str(ROOT))
from lernakademie.build1 import lade_dokumente, suche_relevant

def test_suche_findet_urlaub():
    abschnitte = lade_dokumente()
    treffer = suche_relevant("Wie viele Urlaubstage gibt es?", abschnitte)
    assert treffer, "Urlaubsfrage muss Treffer liefern"
    assert treffer[0]["titel"] == "Urlaub", f"Erster Treffer: {treffer[0]['titel']}"

def test_suche_leer_bei_unbekannt():
    assert suche_relevant("Weihnachtsgeld Hoehe Bonus?", lade_dokumente()) == []

for test in (test_suche_findet_urlaub, test_suche_leer_bei_unbekannt):
    test()          # im Notebook einfach aufrufen – rot = AssertionError
    print("PASS:", test.__name__)

### 🔎 Was sehen wir hier?
Ein `assert` ist ein Test: Stimmt die Behauptung nicht, **crasht** die Zelle mit AssertionError – das ist der Witz, nicht der Fehler! Diese zwei Tests laufen (neben 40+ weiteren) in der **CI** des Repositories (`.github/workflows/tests.yml`) bei jedem Push automatisch – offline, ohne Kosten.

## ✏️ Deine Aufgabe: Deine eigenen zwei Tests

Du schreibst Tests für Funktionen, die DU gebaut hast – und zwar für Grenzfälle.

**Anforderungen:**

- Schreibe zwei Test-Funktionen (mit `assert`) für Bausteine aus deinen Lektionen – z. B.: Build-2-Quellen korrekt (Lektion 09), GraphStore-Nachbarn (Lektion 10) oder die Klassifikation (Lektion 13/15).

- Jeder Test prüft mindestens einen **Grenzfall** (leere Liste, unbekannte Frage, andere thread_id …).

- Rufe beide auf – beide müssen PASS drucken.

**Schreibe deinen Code in die folgende Zelle:**

In [ ]:
def test_1():
    # YOUR CODE HERE: eine Behauptung ueber deinen Assistenten pruefen
    pass

def test_2():
    # YOUR CODE HERE: Grenzfall!
    pass

test_1(); print("PASS: test_1")
test_2(); print("PASS: test_2")

### ✅ Richtig, wenn:

- [ ] Beide Tests laufen grün (kein AssertionError).

- [ ] Mindestens ein Grenzfall ist geprüft.

- [ ] Die Tests nutzen Bausteine aus deinen Lektionen (nicht nur `assert True`).

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [ ]:
test_1()
test_2()
print("Alle Prüfungen bestanden! ✅ (Deine Tests laufen jetzt – das ist echte Ingenieursarbeit.)")

<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Beispiele: `assert lade_dokumente()` / `assert suche_relevant(...) == []` bei Unsinn / `assert store.find_node("Person", name="Anna") is not None`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Ein guter Testname beschreibt die Behauptung: `test_suche_leer_bei_unbekannt`.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Was ist ein Regressionstest im KI-Kontext? (Stichwort: Prompt geändert → Tests/Evals laufen lassen.)

2. Warum ist eine 100%-Erfolgsquote bei zu einfachen Eval-Fragen gefährlich?

## 🧠 Konzept: Evaluierung – die Antwort-Qualität messen

**WAS:** Eine Eval definiert **Testfragen mit erwarteten Eigenschaften** (nicht mit erwartetem Volltext!). Offline ist alles deterministisch – die Quote ist reproduzierbar (Feature!).

In [ ]:
from lernakademie.build2 import beantworte

EVAL_Faelle = [   # (frage, erwartete Eigenschaft als Funktion)
    ("Wie viele Urlaubstage gibt es?",
     lambda e: "mitarbeiter-handbuch.md" in e["quellen"]),
    ("Wie richte ich VPN ein?",
     lambda e: "it-handbuch.md" in e["quellen"]),
    ("Was gilt fuer Passwoerter?",
     lambda e: e["gefundene_stellen"] >= 1),
    ("Wie hoch ist das Weihnachtsgeld?",
     lambda e: e["quellen"] == [] and "leider nicht" in e["antwort"]),
]

treffer = 0
for frage, pruefung in EVAL_Faelle:
    ergebnis = beantworte(frage)
    bestanden = pruefung(ergebnis)
    treffer += int(bestanden)
    print(f"{'PASS' if bestanden else 'FAIL'} | {frage}")
print(f"\nErfolgsquote: {treffer}/{len(EVAL_Faelle)} = {100 * treffer / len(EVAL_Faelle):.0f}%")

## 🧠 Konzept: Kosten & Betrieb in einem Bild

Die drei Betreiber-Regeln, die die Akademie von Anfang an eingebaut hat:
1. **Bremsen:** Loop-Limits (Lektion 13/18), `max_runden` beim Supervisor, Historien-Kürzung (Lektion 04).
2. **Geheimnisse:** nur `.env`, niemals im Code, Keys maskiert in Logs.
3. **Nachvollziehbarkeit:** strukturierte Logs, Audit-Log (Lektion 15/19) – und optional LangSmith (reine Umgebungsvariablen, kein Codeumbau).

**Was vor echter Veröffentlichung noch fehlt:** Login/Auth, HTTPS, Rate-Limits – Ausblick, kein Teil dieser Akademie.

## 🎓 Deine Abschluss-Reflexion (der letzte Auftrag)

Beantworte schriftlich (in der nächsten Zelle) – **in deinen eigenen Worten**:

1. **Wann würdest du heute welche der drei Architekturen bauen?** Nenne je ein konkretes Beispiel aus deinem Leben: ein Fall für Build 1 (reines Python), Build 2 (LangChain + RAG), Build 3 (LangGraph + Graph + Agent).
2. **Warum kann dein Build-3-Assistent Fragen beantworten, die Build 1 und 2 zusammen nicht konnten?** (Stichwort: Emergenz aus Lektion 10/14.)
3. **Was bedeutet "anbieter-unabhängig" konkret?** Was musst du ändern, um von Mistral zu einem lokalen Server zu wechseln – und was NICHT?

Vergleich zum Nachschauen: [docs/same-problem-three-architectures.md](../../docs/same-problem-three-architectures.md)

In [ ]:
# MEINE ABSCHLUSST-REFLEXION:
# 1. Build 1 wuerde ich fuer ... bauen, weil ...
#    Build 2 passt auf ..., Build 3 brauchte ich, wenn ...
# 2. ...
# 3. ...


## 🏆 Geschafft!

Du hast den Weg von **"Was ist ein Terminal?"** bis **"Ich habe ein getestetes Multi-Quellen-Wissenssystem gebaut und über ein Frontend bedient"** zurückgelegt – in 20 Lektionen, drei Builds und einem Finale.

**Dein nächstes Projekt:** Nutze die Bausteine (`lernakademie/`), deine `data/` und das Finale als Vorlage – und bau den Assistenten für **dein** Wissen. Die Werkzeuge hast du jetzt.

*(Deinen Fortschritt kannst du in `PROGRESS.md` abhaken – und der Vergleich der Architekturen lebt in [docs/same-problem-three-architectures.md](../../docs/same-problem-three-architectures.md).)*